In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
root = root.parent
sys.path.insert(0, str(root))

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Dict, Optional
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from types import SimpleNamespace

from sae_trainer.core.model_utils import ReluSparseAutoencoder, TopKSparseAutoencoder, sae_inserted_llm
from sae_trainer.sae_training.train import training_wrapper, get_model, get_dataloader, collect_activations


/Users/alyssagardiner/src/sae_trainer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
cfg = {
    # Model
    'model_name': 'qwen',
    #d_in: 768 # inferred from model
    'expansion_factor': 8,
    'normalize_decoder': True,
    'collection_batch_size': 8,
    'sae_type': 'topk',
    'k': 40,
    'target_layer_idx': 12,

    # Data
    'dataset_name': 'openwebtext',

    # Logging
    'use_wandb': False,
    'wandb_project': 'sae-trainer',

    # Training
    'num_epochs': 5,
    'sae_batch_size': 2048,
    'lr': 3.0e-4,
    'weight_decay': 1.0e-4,
    'lambda_l1': 1.0e-2,
    'lambda_kl': 1.0e-3,
    'target_firing_rate': 0.02,
    'mass_frac_threshold': 0.001,
}

cfg = SimpleNamespace(**cfg)

In [4]:
save_mode = False

device = "mps" if torch.backends.mps.is_available() else "cpu"

llm, tokenizer, collector_class = get_model(cfg, device)

#loader = get_dataloader(cfg, tokenizer)

#collector = collector_class(model=llm)
#target_layers = collector.get_layers()
#collector.register()

#accum = collect_activations(loader, collector, target_layers, device, max_batches=50)
    
#mass_frac_threshold=cfg.mass_frac_threshold

#saes = {}
#histories = {}
#train_loaders = {}
#val_loaders = {}

#for target_layer in target_layers:
#    print(f"Training SAE for layer {target_layer}")
##    sae, history, train_loader, val_loader = training_wrapper(cfg, accum, target_layer, device, mass_frac_threshold, save_mode=save_mode, show_curves=True)
#    saes[target_layer] = sae
#    histories[target_layer] = history
#    train_loaders[target_layer] = train_loader
#    val_loaders[target_layer] = val_loader

In [ ]:
dataset_name = 'openwebtext'
model_name_simple = "qwen" # Either gpt2 or qwen
sae_type = 'topk'

if model_name_simple == "gpt2":
    model_name = "gpt2"
elif model_name_simple == "qwen":
    model_name = "Qwen/Qwen2-0.5B-Instruct"
else:
    raise ValueError(f"Invalid model name: {model_name_simple}. Expected either 'gpt2' or 'qwen'.")
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()

def get_sae(cfg, device):
    k = 40

    # Load SAE checkpoint — pick whichever layer you want to analyse (3, 6, 9, or 11)
    ckpt = torch.load(root / f"model_weights_files/{cfg.sae_type}_sae_{cfg.model_name_simple}_{cfg.dataset_name}_layer{cfg.target_layer_idx}.pt", map_location=device)
    if cfg.sae_type == 'relu':
        sae = ReluSparseAutoencoder(d_in=ckpt["d_in"], d_latent=ckpt["d_latent"], normalize_decoder=True).to(device)
    elif sae_type == 'topk':
        sae = TopKSparseAutoencoder(d_in=ckpt["d_in"], d_latent=ckpt["d_latent"], k=k, normalize_decoder=True).to(device)
    else:
        raise ValueError(f"Invalid sae type: {sae_type}. Expected either 'relu' or 'qwen'.")
    sae.load_state_dict(ckpt["model_state"])
    sae.eval()

    return sae

In [ ]:
# Build openwebtext dataset and FeatureTracer summaries. Keep defaults to reproduce published results

n_train_docs = 4000 # cfg.max_batches * cfg.collection_batch_size

ds = load_dataset("openwebtext", split="train", streaming=True)

train_texts = [
    " ".join(row["text"].split()[:350])  # ~350 words ≈ 512 tokens for English
    for row in ds.shuffle(seed=42, buffer_size=1000)
    if len(row["text"].split()) >= 50
][:n_train_docs]

In [ ]:
sae = get_sae(cfg, device=device)


In [ ]:
from sae_trainer.core.model_utils import sae_regularized_finetune

history = sae_regularized_finetune(
    llm, sae, tokenizer,
    texts=train_texts,
    layer_idx=12,
    target_features={42: 2.0, 107: 0.0},  # amplify 42, suppress 107
    lambda_sae=0.05,
    n_epochs=2,
    batch_size=4,
    eval_sae_drift_every=50,
)
